# CS 691 · Module 06 · Connecting a neural detector to rules

Follow a photograph through a detector and a rule, then locate the first broken link in an incorrect recommendation.

**Before you begin.** You can turn visual features into a score, apply a cutoff, and distinguish a proposed fact from a confirmed one. You can match names in a rule.

The example detector proposes a cap when its score is at least 0.75. If the score is exactly 0.75, what can appear in the record before inspection?

<details><summary>Reveal the answer</summary>

A cap candidate for that photo’s item. “At least” includes equality, and the proposal remains available for inspection.

</details>

## Give the rule a named input

A hybrid system can let a learned component interpret an image and let a symbolic rule decide how to use the result. The two components need an agreement about the meaning of the data they exchange. Here the detector supplies an item ID and a score for a visible cap. A short adapter tests the cutoff and writes a named fact.

The photograph belongs to C1. Its rim and dark-center cues give a score of 0.80. The adapter writes CapCandidate(C1). Carrying the item ID matters when several cameras arrive together.

**Predict before running the cell.** What would the rule lose if the adapter supplied only the word 'candidate'?

In [ ]:
item, rim, dark_center = "C1", 1, 1
score = max(0.0, 0.6 * rim + 0.3 * dark_center - 0.1)
candidates = set()
if score >= 0.75:
    candidates.add(("CapCandidate", item))
print(sorted(candidates))

<details><summary>Reveal output and explanation</summary>

```text
[('CapCandidate', 'C1')]
```

It could no longer connect the photograph to a particular item. A candidate for C1 could be paired with a reservation for C2.

</details>

## Match the same camera in both facts

The fictional desk groups items for a pickup review. Its rule says that if a person reserved an item and the detector proposed a visible cap on that same item, suggest a pickup review for that person and item. A review suggestion invites staff to examine the case.

Reserved(Ava, C1) and CapCandidate(C1) match on C1. Reserved(Ben, C2) cannot use C1's photo. The repeated item name in the rule is the link.

**Predict before running the cell.** If Ben changes his reservation to C1 as well, how many suggestions follow?

In [ ]:
reservations = [("Ava", "C1"), ("Ben", "C2")]
cap_candidates = {"C1"}
for person, item in reservations:
    if item in cap_candidates:
        print("SuggestPickupReview", person, item)

<details><summary>Reveal output and explanation</summary>

```text
SuggestPickupReview Ava C1
```

Two. Both reservations now match C1. The rule can suggest two reviews for one camera. Allocating that camera to a single borrower requires a separate decision.

</details>

## Leave a trail back to the photograph

A proof trace lists the facts and rule behind a derived statement. Add provenance, the origin of each input, so staff can inspect it. The reservation has a record ID. The candidate should retain the photograph ID, detector score and cutoff that produced it.

Start at Ava's review suggestion and work backward. The rule used her reservation R17 for C1. The matching candidate came from photo P9 of C1, scored 0.80 against a cutoff of 0.75.

**Predict before running the cell.** A later inspection finds that P9 shows a reflection. Which part of the trace should change first?

In [ ]:
reservation = ("R17", "Ava", "C1")
photo = ("P9", "C1", 0.80)
reservation_id, person, item = reservation
photo_id, photo_item, score = photo
cutoff = 0.75
if item == photo_item and score >= cutoff:
    print(f"Review {person} {item} via reservation {reservation_id}")
    print(f"Photo {photo_id} scored {score:.2f} against {cutoff:.2f}")

<details><summary>Reveal output and explanation</summary>

```text
Review Ava C1 via reservation R17
Photo P9 scored 0.80 against 0.75
```

Keep P9, its original score and the rule that produced the suggestion. Add the inspector’s finding to the same photo record. Staff can now dismiss this suggestion. A program would need an explicit rule giving the inspection priority before it could dismiss the suggestion automatically.

</details>

## Diagnose a missing recommendation

Suppose Ava's review never appears. Test each boundary in sequence. Did her reservation arrive? Did a photo of her item arrive? Did its score meet the cutoff? A missing input and a low score can produce the same empty queue, but each calls for a different fix.

The desk received Ava's C1 reservation and a photo of C2 scoring 0.92. The high score concerns C2. Asking for a C1 photograph addresses the first missing piece.

**Predict before running the cell.** What would the program report if a C1 photograph arrived and scored 0.68?

In [ ]:
reserved_item = "C1"
photos = {"C2": 0.92}
if reserved_item not in photos:
    reason = "Missing photo for " + reserved_item
elif photos[reserved_item] < 0.75:
    reason = "Inspect low-scoring photo"
else:
    reason = "Suggest pickup review"
print(reason)

<details><summary>Reveal output and explanation</summary>

```text
Missing photo for C1
```

It would report 'Inspect low-scoring photo'. The identifier now matches, but 0.68 falls below 0.75. Taking another C2 photo would not address that result.

</details>

## Check for the right photo before scoring it

The dictionary pairs each photographed camera with its score. The first condition asks if the reserved camera is missing from that dictionary. Python evaluates the score comparison only when it has found a photo for the reserved camera.

Replace this line:
photos = {"C2": 0.92}
with:
photos = {"C2": 0.92, "C1": 0.75}
What will print?

In [ ]:
reserved_item = "C1"
photos = {"C2": 0.92}
if reserved_item not in photos:
    reason = "Missing photo for " + reserved_item
elif photos[reserved_item] < 0.75:
    reason = "Inspect low-scoring photo"
else:
    reason = "Suggest pickup review"
print(reason)

<details><summary>Worked answer after your change</summary>

Suggest pickup review

C1’s new photo lands exactly on the cutoff. The comparison asks whether the score is below 0.75. Equality reaches the final branch and sends the case for review.

</details>

## Exercise 1

Fill the missing variable in the rule 'Reserved(person, item) AND CapCandidate(?) implies SuggestPickupReview(person, item)'. Give a case that explains your choice.

<details><summary>Hint</summary>

Look for the identifier shared by the photo and reservation.

</details>

<details><summary>Worked answer</summary>

Use item. If Ava reserved C1 and only C2 has a candidate, using a different variable would let the C2 photo trigger a C1 review.

</details>

## Exercise 2

Ava reserved C1 and C3. Photos score C1 = 0.75, C2 = 0.97 and C3 = 0.74. List her reviews at an inclusive 0.75 cutoff.

<details><summary>Hint</summary>

Filter the photos first, then match the item names in Ava's two reservations.

</details>

<details><summary>Worked answer</summary>

C1 and C2 clear the cutoff. Only C1 is in Ava's reservations, so the rule suggests one review for Ava and C1.

</details>

## Exercise 3

Write the minimum evidence you would request to reproduce a disputed review for C2.

<details><summary>Hint</summary>

Think about both inputs, the adapter, and the matching rule.

</details>

<details><summary>Worked answer</summary>

Ask for the reservation and its item ID, the photo and its item ID, the recorded score and cutoff, and the rule version used to join them. These let you check whether each premise was available and whether the identifiers matched.

</details>

So far, the rule acts after the detector produces a score. Can knowledge from the desk also influence the weights the detector learns?

Staff repeatedly confirm caps on photos that score below the cutoff. The desk wants future scores on photos like these to rise during training. Where in the system would that information need to enter?

<details><summary>Reveal the answer</summary>

It would have to affect how candidate detector weights are judged during training. A desk rule that runs only after scoring can change the review decision while leaving the learned scores untouched.

</details>